In [ ]:
#Model 4: Prediction the rating with Multinomial Naive Bayes model

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

df = pd.read_csv("clean_reviews.csv")

required = {"review_text", "rating"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Mangler kolonner: {missing}. Fant: {list(df.columns)}")

df = df.dropna(subset=["review_text", "rating"]).copy()
df["review_text"] = df["review_text"].astype(str)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating"]).copy()
df["rating"] = df["rating"].astype(int)
df = df[(df["rating"] >= 1) & (df["rating"] <= 5)].copy()

# fjern verdien 3
df = df[df["rating"] != 3].copy()

# binær-klassifisering: 1-2 = 0 (negativ), 4-5 = 1 (positiv)
df["sentiment"] = df["rating"].apply(lambda x: 0 if x <= 2 else 1)

X = df["review_text"]
y = df["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if y.nunique() > 1 else None
)

nb_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,1), max_features=5000, stop_words="english")),
    ("nb", MultinomialNB(alpha=0.5))
])

# Train
nb_model.fit(X_train, y_train)

# Predict
y_pred = nb_model.predict(X_test)

print("\n--- Classification report (TEST) ---")
print(classification_report(y_test, y_pred, digits=3))

print("\nAccuracy (TEST):", accuracy_score(y_test, y_pred))

print("\n--- Confusion matrix (TEST) ---")
print(confusion_matrix(y_test, y_pred))

# Retrain on full dataset for saving predictions
nb_model.fit(X, y)
df["predicted_sentiment"] = nb_model.predict(X)

cols = list(df.columns)
cols.remove("predicted_sentiment")
score_idx = cols.index("rating")
cols.insert(score_idx + 1, "predicted_sentiment")
df = df[cols]

print("\nFerdig. Lagret som:", path_out)
print(df[["rating", "sentiment", "predicted_sentiment"]].head())


--- Classification report (TEST) ---
              precision    recall  f1-score   support

           1      0.646     0.594     0.619       283
           2      0.600     0.043     0.080        70
           4      0.333     0.053     0.091        95
           5      0.706     0.914     0.796       569

    accuracy                          0.684      1017
   macro avg      0.571     0.401     0.397      1017
weighted avg      0.647     0.684     0.632      1017


Accuracy (TEST): 0.6843657817109144

--- Confusion matrix (TEST) ---
[[168   0   2 113]
 [ 32   3   4  31]
 [ 16   1   5  73]
 [ 44   1   4 520]]

Ferdig. Lagret som: (Model 4) reviews_naive_bayes.csv
   rating  sentiment  predicted_sentiment
0       5          1                    5
1       5          1                    5
2       5          1                    5
3       5          1                    5
4       4          1                    5
